<a href="https://colab.research.google.com/github/kuds/courtside-dynamics/blob/main/notebooks/spikes/s1_gpu_throughput.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spike S1: can the humanoid track train on Colab GPUs?

Design `docs/design_env_ladder_and_structure_20261016.md`, section 5
("Spikes"): the humanoid rungs (H0 to H4) train on GPU only, and they
start only after this spike passes two numeric bars on a Colab L4.

| Bar | Measured on | Pass |
|---|---|---|
| **Single G1** | MuJoCo Playground's Unitree G1 joystick env (and mjlab's G1 velocity task, when mjlab installs), batched and compiled: env steps/s | **>= 2x10^4 steps/s** |
| **4xG1 + ball** | this repo's G1 (`assets/robots/unitree_g1_tennis.xml`) four times plus the packaged tennis ball, composed with MjSpec, on MJX: control steps/s with physics at 1 kHz | **>= 5x10^3 steps/s** |

**What a step is.** Both bars count control (env) steps per second, with
compile time and warm-up calls off the clock.
- Single G1: one env `step` per env. Playground's G1 runs 50 Hz control
  over 500 Hz physics (10 substeps), inside Brax's training wrappers
  (episodes, auto-reset, domain randomization), with random actions.
  mjlab's runs 50 Hz over 200 Hz (4 substeps) with its managers.
- 4xG1 + ball: one 50 Hz control period of the whole scene, 20 physics
  steps at 1 kHz (the low end of the design's 1-2 kHz), with random joint
  targets around the standing pose and no env logic. The bar is then
  4 seats x 5x10^3 = 2x10^4 agent steps/s, the single-G1 rate per agent.
  Physics steps/s are reported beside it.
- Neither rate includes a policy forward pass or a PPO update. Training
  runs slower than these rates, so a failed bar fails training too.

**Reading.** Each bar takes its best valid configuration (stack, MJX
implementation, batch size). A configuration is reported but excluded
when it errors, runs out of memory, overflows a MuJoCo Warp contact or
constraint buffer, or ends with non-finite state. `SMOKE = True` checks
every code path on a CPU runtime at toy sizes and never yields a verdict.

**Next step**, printed by section 7:
- both bars pass on the L4: plan H0 (G1Footwork) on Colab L4 GPUs;
- a bar fails on the L4: re-run this notebook unchanged on a rented
  A100/4090-class GPU. If that run fails too, pause the humanoid track;
  the paddle rungs continue on CPU.

1. **Install** pinned versions.
2. **Settings**, the **environment** record (GPU, CUDA, versions) and
   the results **folder**.
3. **mjlab** (optional), in its own venv. It runs first, before JAX
   holds GPU memory.
4. **Playground** single G1 at batch 1024, 4096 and 8192, on both MJX
   implementations: `warp` (MJX on MuJoCo Warp, Playground's G1 default)
   and `jax` (classic MJX).
5. **4xG1 + ball**: compose with MjSpec, convert to MJX, batch-step. If
   MJX rejects the model, the rejection is reported verbatim and the
   scene falls back to a simplified collision set; each row records which
   set it measured.
6. **Results**: a JSON in Drive `diagnostics/` (never `training_runs/`),
   saved after every section.
7. **Verdict**: PASS or FAIL per bar and the next step.

Budget: the defaults are sized for about 45 minutes on an L4. That is an
estimate, since the CPU smoke run checks code paths, not GPU timings;
each section prints its wall time, and a sweep starts no new
configuration after `SECTION_BUDGET_MIN`. The design allows about 10
L4-hours for S1. Runtime: **Runtime > Change runtime type > L4 GPU**.

## 1. Install

In [ ]:
# Immutable git ref to install: a full commit SHA (or a dated tag such as
# `name-YYYYMMDD`), never a branch name, so a merge to `main` cannot change
# what a session measures; tests/test_spike_notebooks.py enforces the pin.
# The spike reads only the packaged Unitree G1 adapter
# (assets/robots/unitree_g1_tennis.xml and its third_party meshes), the
# packaged court's ball and contact parameters (assets/humanoid_tennis.xml)
# and the standing reference in envs/robot_models.py, all unchanged since
# this ref.
REPO_REF = "b9585e2961591022617223eed3e809bf5d9d1d0f"

# The GPU stack, pinned together; the results record every version.
# mujoco 3.15.0 is the newest the package supports (it caps <3.16 from
# the week-1 refs on) and mujoco-mjx must match it. jax[cuda12] brings
# its own CUDA 12 libraries: a driver too old for them leaves JAX on the
# CPU, which section 2 refuses.
!pip install -q "jax[cuda12]==0.11.2" "mujoco==3.15.0" "mujoco-mjx==3.15.0" "warp-lang==1.18.0" "playground==0.2.0" "brax==0.14.2"
!pip install -q "courtside-dynamics @ git+https://github.com/kuds/courtside-dynamics@{REPO_REF}"
print(f"Installed courtside-dynamics from ref: {REPO_REF}")

## 2. Settings, environment and folder

The bars are fixed by the design. The rest sizes the sweep: what to
measure, how long each measurement runs, and where the results go.
`SMOKE = True` shrinks everything to batch 2 and one timed call.

In [ ]:
import os

# SMOKE runs every code path at toy sizes (batch 2, one timed call) so a
# CPU runtime can check the notebook end to end. Its numbers measure
# nothing: a SMOKE run never yields a verdict.
SMOKE = False

USE_DRIVE = True
# The results folder, relative to the courtside-dynamics Drive root
# (MyDrive/Finding Theta/courtside-dynamics/). A folder inside
# training_runs/ is refused.
RESULTS_SUBDIR = "diagnostics/s1_gpu_throughput"

# The bars (design section 5, spike S1). Not knobs.
SINGLE_G1_BAR = 2e4  # single-G1 env steps/s
FOUR_G1_BAR = 5e3  # 4xG1 + ball control steps/s, physics at PHYSICS_HZ

# Section 3: mjlab (MuJoCo Warp + rsl_rl), optional. It pins mujoco 3.11,
# so it gets its own venv (sharing this runtime's torch) and runs in a
# subprocess.
RUN_MJLAB = True
MJLAB_PIN = "mjlab==1.6.0"
MJLAB_TASK = "Mjlab-Velocity-Flat-Unitree-G1"
MJLAB_NUM_ENVS = (1024, 4096, 8192)
MJLAB_VENV = "/content/mjlab-venv"
MJLAB_WARMUP_STEPS = 20  # mjlab builds its kernels and CUDA graphs here

# Section 4: one G1 on MuJoCo Playground.
PLAYGROUND_ENV = "G1JoystickFlatTerrain"
# "warp": MJX on MuJoCo Warp, Playground's G1 default; "jax": classic MJX.
PLAYGROUND_IMPLS = ("warp", "jax")
PLAYGROUND_BATCH_SIZES = (1024, 4096, 8192)
PLAYGROUND_DOMAIN_RANDOMIZATION = True  # as Playground's G1 PPO trains
# MuJoCo Warp's contact buffer is shared by every world: the G1 default
# allots 8 per env (8 x 8192).
PLAYGROUND_CONTACTS_PER_ENV = 8
UNROLL = 20  # env steps per compiled call: Brax PPO's G1 unroll length

# Section 5: four G1s and a ball.
SCENE_IMPLS = ("warp", "jax")
SCENE_BATCH_SIZES = (1024, 4096, 8192)
PHYSICS_HZ = 1000  # the bar's rate: the low end of the design's 1-2 kHz
CONTROL_HZ = 50  # the G1 tracker's rate in the design's control hierarchy
EXTRA_PHYSICS_HZ = (2000,)  # informational, at the bar's best (impl, batch)
INTEGRATOR = "implicitfast"  # mjlab's default; the CPU env runs RK4
SOLVER_ITERATIONS = 10  # mjlab's G1 velocity-task caps (Playground's G1: 3, 5)
LS_ITERATIONS = 20
ACTION_SCALE = 0.5  # random targets: standing pose + U(-1, 1) x this, rad
SCENE_UNROLL = 5  # control steps per compiled call
# MuJoCo Warp capacity per world. A CPU MuJoCo rollout of this scene
# (4 s, six seeds, every robot falling) peaked at 80 contacts and 436
# constraint rows. An overflow invalidates a row; it never passes one.
SCENE_CONTACTS_PER_WORLD = 160
SCENE_CONSTRAINTS_PER_WORLD = 900
# "auto": the full collision set, simplified only if MJX rejects it.
# "always": the simplified set, to measure or check that path.
SIMPLIFY_COLLISION = "auto"

# Timing, every section.
WARMUP_CALLS = 2  # executions after compiling, before the clock starts
MIN_TIMED_S = 10.0  # timed calls run until this long has passed ...
MAX_TIMED_CALLS = 50  # ... or this many calls
SECTION_BUDGET_MIN = 15  # a sweep starts no new configuration after this
SEED = 0

if SMOKE:
    MJLAB_NUM_ENVS = (2,)
    MJLAB_WARMUP_STEPS = 1
    PLAYGROUND_BATCH_SIZES = (2,)
    SCENE_BATCH_SIZES = (2,)
    UNROLL = 2
    SCENE_UNROLL = 1
    WARMUP_CALLS = 1
    MIN_TIMED_S = 0.0
    MAX_TIMED_CALLS = 1

if not 1000 <= PHYSICS_HZ <= 2000:
    raise ValueError(f"PHYSICS_HZ = {PHYSICS_HZ}: the 4xG1 bar is defined at 1-2 kHz")
for hz in (PHYSICS_HZ, *EXTRA_PHYSICS_HZ):
    if hz % CONTROL_HZ:
        raise ValueError(f"{hz} Hz physics is not a whole number of steps per {CONTROL_HZ} Hz control step")
if SIMPLIFY_COLLISION not in ("auto", "always"):
    raise ValueError(f"SIMPLIFY_COLLISION = {SIMPLIFY_COLLISION!r}: 'auto' or 'always'")
for impl in (*PLAYGROUND_IMPLS, *SCENE_IMPLS):
    if impl not in ("warp", "jax"):
        raise ValueError(f"unknown MJX implementation {impl!r}: 'warp' or 'jax'")

# Before JAX loads: allocate GPU memory on demand, so MuJoCo Warp and the
# mjlab subprocess find room beside JAX, and add the XLA flag Playground
# recommends on NVIDIA GPUs.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
if "--xla_gpu_triton_gemm_any" not in os.environ.get("XLA_FLAGS", ""):
    os.environ["XLA_FLAGS"] = (os.environ.get("XLA_FLAGS", "") + " --xla_gpu_triton_gemm_any=True").strip()

In [ ]:
import json
import platform
import re
import subprocess
import sys
import time
from datetime import UTC, datetime
from importlib.metadata import PackageNotFoundError, version

import jax

STARTED = time.monotonic()
STARTED_UTC = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")


def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def nvidia_smi():
    """Name, driver, memory and the driver's CUDA version; None without a GPU."""
    try:
        query = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout
        banner = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True).stdout
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None
    name, driver, memory = (field.strip() for field in query.splitlines()[0].split(","))
    cuda = re.search(r"CUDA Version:\s*([0-9.]+)", banner)
    return {"name": name, "driver": driver, "memory": memory, "driver_cuda": cuda.group(1) if cuda else None}


BACKEND = jax.default_backend()
if BACKEND != "gpu" and not SMOKE:
    raise RuntimeError(
        f"JAX runs on {BACKEND}, not a GPU, and S1 measures GPU throughput: pick an L4 "
        "runtime (or set SMOKE = True to check the code paths on CPU)"
    )
GPU_NAME = jax.devices()[0].device_kind if BACKEND == "gpu" else "cpu"
ENVIRONMENT = {
    "gpu": GPU_NAME,
    "nvidia_smi": nvidia_smi(),
    "jax_backend": BACKEND,
    "jax_devices": [str(device) for device in jax.devices()],
    "x64": bool(jax.config.jax_enable_x64),
    "xla_flags": os.environ.get("XLA_FLAGS", ""),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "repo_ref": REPO_REF,
    "versions": {
        name: package_version(name)
        for name in (
            "jax", "jaxlib", "jax-cuda12-plugin", "jax-cuda12-pjrt", "nvidia-cuda-runtime-cu12",
            "nvidia-cudnn-cu12", "mujoco", "mujoco-mjx", "warp-lang", "playground", "brax", "flax",
            "numpy", "courtside-dynamics",
        )
    },
}
print(json.dumps(ENVIRONMENT, indent=2))

In [ ]:
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Finding Theta/courtside-dynamics")
else:
    PROJECT_ROOT = Path(".")
TRAINING_RUNS = PROJECT_ROOT / "training_runs"
RESULTS_DIR = PROJECT_ROOT / RESULTS_SUBDIR
if RESULTS_DIR.resolve().is_relative_to(TRAINING_RUNS.resolve()):
    raise ValueError(f"{RESULTS_DIR} lies inside {TRAINING_RUNS}: S1 results never go there")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
GPU_SLUG = re.sub(r"[^a-z0-9]+", "-", GPU_NAME.lower()).strip("-")
RESULTS_PATH = RESULTS_DIR / f"s1_{'smoke_' if SMOKE else ''}{GPU_SLUG}_{STARTED_UTC}.json"
SETTINGS = {
    name: globals()[name]
    for name in (
        "SMOKE", "RUN_MJLAB", "MJLAB_PIN", "MJLAB_TASK", "MJLAB_NUM_ENVS", "MJLAB_WARMUP_STEPS",
        "PLAYGROUND_ENV", "PLAYGROUND_IMPLS", "PLAYGROUND_BATCH_SIZES",
        "PLAYGROUND_DOMAIN_RANDOMIZATION", "PLAYGROUND_CONTACTS_PER_ENV", "UNROLL", "SCENE_IMPLS",
        "SCENE_BATCH_SIZES", "PHYSICS_HZ", "CONTROL_HZ", "EXTRA_PHYSICS_HZ", "INTEGRATOR",
        "SOLVER_ITERATIONS", "LS_ITERATIONS", "ACTION_SCALE", "SCENE_UNROLL",
        "SCENE_CONTACTS_PER_WORLD", "SCENE_CONSTRAINTS_PER_WORLD", "SIMPLIFY_COLLISION",
        "WARMUP_CALLS", "MIN_TIMED_S", "MAX_TIMED_CALLS", "SECTION_BUDGET_MIN", "SEED",
    )
}
RESULTS = {
    "spike": "S1",
    "design": "docs/design_env_ladder_and_structure_20261016.md, section 5",
    "notebook": "notebooks/spikes/s1_gpu_throughput.ipynb",
    "smoke": SMOKE,
    "started_utc": STARTED_UTC,
    "environment": ENVIRONMENT,
    "bars": {"single_g1_env_steps_per_s": SINGLE_G1_BAR, "four_g1_control_steps_per_s": FOUR_G1_BAR},
    "settings": SETTINGS,
    "mjlab": None,
    "playground": None,
    "scene": None,
    "verdict": None,
}


def save_results():
    """Write RESULTS whole (temporary file, then rename); every section calls this."""
    RESULTS["saved_utc"] = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    RESULTS["wall_s"] = time.monotonic() - STARTED
    text = json.dumps(RESULTS, indent=2, allow_nan=False) + "\n"
    partial = RESULTS_PATH.with_name(RESULTS_PATH.name + ".partial")
    partial.write_text(text)
    os.replace(partial, RESULTS_PATH)
    return RESULTS_PATH


print("Results:", save_results())

### How the verdict reads the rows

Every measurement is one row. A row counts toward its bar only when it
was measured at the bar's settings and cleanly: no error, no MuJoCo Warp
buffer overflow, no non-finite state. Each bar takes its best counting
row. The next step follows the design: the L4 run either clears H0 for
Colab GPUs or sends the notebook to a rented GPU, and a rented run that
fails pauses the humanoid track.

In [ ]:
import re


def counts(row):
    """The row was measured at the bar's settings, and cleanly."""
    return (
        bool(row.get("bar"))
        and row.get("steps_per_s") is not None
        and not row.get("error")
        and not row.get("overflow")
        and not row.get("nonfinite_worlds")
    )


def best_row(rows):
    """The fastest counting row, or None."""
    candidates = [row for row in rows if counts(row)]
    return max(candidates, key=lambda row: row["steps_per_s"]) if candidates else None


def is_l4(gpu_name):
    return re.search(r"\bL4\b", gpu_name) is not None


def s1_verdict(single_rows, scene_rows, *, single_bar, four_bar, gpu_name, smoke):
    """Each bar's best row and status, the overall reading, and the design's next step."""
    names = {"single_g1": "single G1", "four_g1": "4xG1 + ball"}
    bars = {}
    for key, rows, bar in (("single_g1", single_rows, single_bar), ("four_g1", scene_rows, four_bar)):
        best = best_row(rows)
        if best is None:
            status = "NO RESULT"
        elif smoke:
            status = "SMOKE"
        else:
            status = "PASS" if best["steps_per_s"] >= bar else "FAIL"
        bars[key] = {"bar": bar, "status": status, "best": best}
    statuses = {key: entry["status"] for key, entry in bars.items()}
    stacks = {
        stack: best_row([row for row in single_rows if row.get("stack") == stack])
        for stack in ("mjlab", "playground")
    }
    missing = [names[key] for key, status in statuses.items() if status == "NO RESULT"]
    failed = [names[key] for key, status in statuses.items() if status == "FAIL"]
    excluded = sum(1 for row in (*single_rows, *scene_rows) if row.get("bar") and not counts(row))
    if missing:
        overall = "NO RESULT"
        next_step = (
            f"No valid measurement for {' and '.join(missing)}: fix the errors reported above "
            "and re-run. S1 has no verdict yet."
        )
    elif smoke:
        overall = "SMOKE"
        next_step = (
            "A SMOKE run checks the code paths and decides nothing: re-run on an L4 runtime "
            "with SMOKE = False."
        )
    elif not failed:
        overall = "PASS"
        mjlab = stacks["mjlab"]
        if mjlab is not None and mjlab["steps_per_s"] >= single_bar:
            stack = "mjlab, the design's first choice, clears the single-G1 bar."
        else:
            stack = (
                "mjlab did not clear the single-G1 bar here ("
                + ("not measured" if mjlab is None else f"best {mjlab['steps_per_s']:,.0f} steps/s")
                + "), so design section 10's fallback applies: Playground/Brax."
            )
        if is_l4(gpu_name):
            next_step = f"Both bars pass on the Colab L4: plan H0 (G1Footwork) on Colab L4 GPUs. {stack}"
        else:
            next_step = (
                f"Both bars pass on {gpu_name}, not an L4: the humanoid track trains on this GPU "
                f"class. Rent one for the humanoid track only (design section 10) and plan H0 "
                f"on it. {stack}"
            )
    else:
        overall = "FAIL"
        if is_l4(gpu_name):
            next_step = (
                f"{' and '.join(failed)} fail on the Colab L4: re-run this notebook unchanged on "
                "a rented A100/4090-class GPU. If that run also fails, pause the humanoid track; "
                "the paddle rungs continue on CPU."
            )
        else:
            next_step = (
                f"{' and '.join(failed)} fail on {gpu_name}, the rented re-run: pause the "
                "humanoid track (design section 5). The paddle rungs continue on CPU."
            )
        if excluded:
            next_step += (
                f" First check the {excluded} bar configuration(s) that did not count (section 6):"
                " a configuration that errored or was skipped could have passed."
            )
    return {"gpu": gpu_name, "bars": bars, "overall": overall, "next_step": next_step, "excluded_rows": excluded}

### Timing

Each configuration is compiled ahead of time, run `WARMUP_CALLS` times
(MuJoCo Warp builds its kernels on the first executions), and then timed
over whole compiled calls until `MIN_TIMED_S` has passed. A compiled
call refuses an input of another type, so a state that drifts fails
loudly instead of recompiling on the clock. One configuration's error is
reported as its row and the sweep goes on, except in a SMOKE run, which
stops on the first error.

In [ ]:
import functools
import gc
import traceback

import jax.numpy as jnp
import mujoco
import numpy as np
import warp as wp
from mujoco import mjx
from mujoco.mjx.third_party.mujoco_warp import OverflowType

# Warp builds for one CUDA toolkit; a driver too old for it leaves Warp
# without a GPU. The record says which, and a warp row on a GPU runtime
# refuses to run on the CPU.
ENVIRONMENT["warp"] = {
    "cuda_devices": wp.get_cuda_device_count(),
    "cuda_driver": wp.get_cuda_driver_version(),
    "cuda_toolkit": wp.get_cuda_toolkit_version(),
}
print("Warp:", ENVIRONMENT["warp"])

# Overflow bits meaning MuJoCo Warp dropped contacts or constraint rows.
# The two iteration bits only say a capped solver stopped at its cap,
# which both GPU stacks do by design; rows count them separately.
CAP_BITS = OverflowType.ITERATIONS | OverflowType.LS_ITERATIONS
DROP_FLAGS = [flag for flag in OverflowType if flag not in (OverflowType.ALL, 0) and not flag & CAP_BITS]


def require_warp_gpu(impl):
    if impl == "warp" and BACKEND == "gpu" and not ENVIRONMENT["warp"]["cuda_devices"]:
        raise RuntimeError(f"Warp sees no CUDA device ({ENVIRONMENT['warp']}): this warp row would not run on the GPU")


def strong(tree):
    """Strip weak types, so a compiled call's output feeds it back unchanged."""
    return jax.tree.map(lambda x: x.astype(x.dtype), tree)


def quiet(mx):
    """Stop MuJoCo Warp printing overflows from its kernels, once per world and step.

    Print-only: the rows read the overflow bitmask instead."""
    if mx.impl.value != "warp":
        return mx
    return mx.replace(opt=mx.opt.replace(_impl=mx.opt._impl.replace(warn_overflow=0)))


def health(data):
    """Non-finite worlds, MuJoCo Warp overflows per flag, and the constraint sizes behind the cost.

    Classic MJX evaluates every candidate contact slot each step, so it
    has fixed sizes and no buffer to overflow; MuJoCo Warp sizes rows by
    the contacts it finds, inside the capacity it was given."""
    fields = {"nonfinite_worlds": int(np.sum(~np.isfinite(np.asarray(data.qpos)).all(axis=-1)))}
    bits = getattr(data._impl, "overflow", None)
    if bits is None:
        return {
            **fields, "overflow": {}, "solver_cap_worlds": None,
            "contact_slots": int(data._impl.ncon), "constraint_rows": int(data._impl.nefc),
        }
    fields["constraint_rows_max_at_end"] = int(np.max(np.asarray(data._impl.nefc)))
    bits = np.asarray(bits)
    fields["overflow"] = {flag.name: int(np.sum((bits & flag) != 0)) for flag in DROP_FLAGS if np.any(bits & flag)}
    fields["solver_cap_worlds"] = int(np.sum((bits & int(CAP_BITS)) != 0))
    return fields


def timed(fn, state):
    """Compile fn(state, key) -> state, warm it up, then time whole calls."""
    started = time.perf_counter()
    compiled = jax.jit(fn).lower(state, jax.random.PRNGKey(SEED)).compile()
    compile_s = time.perf_counter() - started
    keys = jax.random.split(jax.random.PRNGKey(SEED + 1), WARMUP_CALLS + MAX_TIMED_CALLS)
    started = time.perf_counter()
    for call in range(WARMUP_CALLS):
        state = compiled(state, keys[call])
    jax.block_until_ready(state)
    warmup_s = time.perf_counter() - started
    calls, started = 0, time.perf_counter()
    while calls < MAX_TIMED_CALLS and (calls == 0 or time.perf_counter() - started < MIN_TIMED_S):
        state = compiled(state, keys[WARMUP_CALLS + calls])
        jax.block_until_ready(state)
        calls += 1
    timed_s = time.perf_counter() - started
    return state, {"compile_s": compile_s, "warmup_s": warmup_s, "timed_calls": calls, "timed_s": timed_s}


def sweep(section, configs, measure):
    """measure(**config) per config, in order; returns the rows and the wall time."""
    rows, started = [], time.monotonic()
    for config in configs:
        label = " ".join(f"{key}={value}" for key, value in config.items())
        if time.monotonic() - started > 60 * SECTION_BUDGET_MIN:
            rows.append({**config, "steps_per_s": None, "error": f"skipped: past SECTION_BUDGET_MIN = {SECTION_BUDGET_MIN}"})
            print(f"{section} {label}: SKIPPED, past the section budget")
            continue
        try:
            row = measure(**config)
        except Exception as error:  # one configuration's failure is its row
            if SMOKE:
                raise
            traceback.print_exc()
            row = {**config, "steps_per_s": None, "error": f"{type(error).__name__}: {error}"[:4000]}
            print(f"{section} {label}: ERROR {row['error'][:400]}")
        else:
            flags = [
                f"OVERFLOW {row['overflow']}" if row.get("overflow") else "",
                f"NON-FINITE in {row['nonfinite_worlds']} worlds" if row.get("nonfinite_worlds") else "",
            ]
            print(
                f"{section} {label}: {row['steps_per_s']:,.0f} steps/s (compile {row['compile_s']:.0f} s, "
                f"warm-up {row['warmup_s']:.1f} s, {row['timed_calls']} calls in {row['timed_s']:.1f} s) "
                + " ".join(flag for flag in flags if flag)
            )
        rows.append(row)
        jax.clear_caches()
        gc.collect()
    return rows, time.monotonic() - started

## 3. mjlab (optional)

mjlab (MuJoCo Warp + PyTorch + rsl_rl) is the design's first-choice GPU
stack. It pins `mujoco` 3.11, so it installs into its own venv that
shares this runtime's PyTorch, and runs its G1 velocity task in a
subprocess with random actions. It runs before the JAX sections so its
process finds the GPU memory free. Any install or run failure is printed
and recorded; the bars then rest on Playground.

In [ ]:
%%writefile s1_mjlab_bench.py
"""S1 spike: mjlab G1 velocity-task throughput; runs in mjlab's own venv."""

import argparse
import json
import platform
import time
from importlib.metadata import version

import torch

import mjlab.tasks  # noqa: F401  (registers the tasks)
from mjlab.envs import ManagerBasedRlEnv
from mjlab.tasks.registry import load_env_cfg

parser = argparse.ArgumentParser()
parser.add_argument("--task", required=True)
parser.add_argument("--device", required=True)
parser.add_argument("--num-envs", type=int, nargs="+", required=True)
parser.add_argument("--warmup-steps", type=int, required=True)
parser.add_argument("--min-timed-s", type=float, required=True)
parser.add_argument("--max-timed-steps", type=int, required=True)
parser.add_argument("--chunk", type=int, required=True)
parser.add_argument("--budget-s", type=float, required=True)
args = parser.parse_args()
cuda = args.device.startswith("cuda")


def sync():
    if cuda:
        torch.cuda.synchronize()


def measure(num_envs):
    cfg = load_env_cfg(args.task)
    cfg.scene.num_envs = num_envs
    started = time.perf_counter()
    env = ManagerBasedRlEnv(cfg=cfg, device=args.device)
    env.reset()
    sync()
    build_s = time.perf_counter() - started
    try:
        dim = env.action_manager.total_action_dim
        generator = torch.Generator(device=args.device).manual_seed(0)

        def step():
            env.step(torch.rand((num_envs, dim), device=args.device, generator=generator) * 2 - 1)

        started = time.perf_counter()
        for _ in range(args.warmup_steps):
            step()
        sync()
        warmup_s = time.perf_counter() - started
        steps, started = 0, time.perf_counter()
        while steps < args.max_timed_steps and (steps == 0 or time.perf_counter() - started < args.min_timed_s):
            for _ in range(args.chunk):
                step()
            sync()
            steps += args.chunk
        timed_s = time.perf_counter() - started
        return {
            "num_envs": num_envs,
            "env_steps_per_s": num_envs * steps / timed_s,
            "timed_steps": steps,
            "timed_s": timed_s,
            "warmup_steps": args.warmup_steps,
            "warmup_s": warmup_s,
            "build_s": build_s,
            "physics_dt": env.physics_dt,
            "control_dt": env.step_dt,
            "substeps": round(env.step_dt / env.physics_dt),
            "solver_iterations": int(env.sim.mj_model.opt.iterations),
            "ls_iterations": int(env.sim.mj_model.opt.ls_iterations),
            "action_dim": dim,
        }
    finally:
        env.close()
        del env
        if cuda:
            torch.cuda.empty_cache()


rows, started = [], time.monotonic()
for num_envs in args.num_envs:
    if time.monotonic() - started > args.budget_s:
        rows.append({"num_envs": num_envs, "error": "skipped: past the section budget"})
        continue
    try:
        rows.append(measure(num_envs))
        print(f"mjlab num_envs={num_envs}: {rows[-1]['env_steps_per_s']:,.0f} env steps/s", flush=True)
    except Exception as error:  # one size's failure is its row; the parent prints it
        rows.append({"num_envs": num_envs, "error": f"{type(error).__name__}: {error}"[:4000]})
        print(f"mjlab num_envs={num_envs}: ERROR {rows[-1]['error'][:400]}", flush=True)

result = {
    "task": args.task,
    "device": args.device,
    "gpu": torch.cuda.get_device_name(0) if cuda else None,
    "python": platform.python_version(),
    "versions": {
        name: version(name) for name in ("mjlab", "mujoco", "mujoco-warp", "warp-lang", "torch", "rsl-rl-lib")
    },
    "rows": rows,
}
print("S1_MJLAB_RESULT " + json.dumps(result), flush=True)

In [ ]:
def tail(text, lines=40):
    return "\n".join(text.strip().splitlines()[-lines:])


def mjlab_section():
    """Install mjlab into its own venv and benchmark it there. Failures are returned, not raised."""
    venv_python = Path(MJLAB_VENV) / "bin" / "python"
    commands = []
    if not venv_python.exists():
        commands += [
            [sys.executable, "-m", "pip", "install", "-q", "virtualenv"],
            [sys.executable, "-m", "virtualenv", "-q", "--system-site-packages", MJLAB_VENV],
        ]
    commands.append([str(venv_python), "-m", "pip", "install", "-q", MJLAB_PIN])
    started = time.monotonic()
    for command in commands:
        try:
            done = subprocess.run(command, capture_output=True, text=True, timeout=30 * 60)
        except subprocess.TimeoutExpired:
            return {"status": "install failed", "command": " ".join(command), "output": "timed out after 30 min"}
        if done.returncode != 0:
            return {"status": "install failed", "command": " ".join(command), "output": tail(done.stdout + done.stderr)}
    install_s = time.monotonic() - started
    command = [
        str(venv_python), "s1_mjlab_bench.py",
        "--task", MJLAB_TASK,
        "--device", "cuda:0" if BACKEND == "gpu" else "cpu",
        "--num-envs", *(str(n) for n in MJLAB_NUM_ENVS),
        "--warmup-steps", str(MJLAB_WARMUP_STEPS),
        "--min-timed-s", str(MIN_TIMED_S),
        "--max-timed-steps", str(MAX_TIMED_CALLS * UNROLL),
        "--chunk", str(UNROLL),
        "--budget-s", str(60 * SECTION_BUDGET_MIN),
    ]
    started = time.monotonic()
    try:
        done = subprocess.run(
            command, capture_output=True, text=True, timeout=60 * (SECTION_BUDGET_MIN + 30),
            env={**os.environ, "MJLAB_WARP_QUIET": "1"},
        )
    except subprocess.TimeoutExpired:
        return {"status": "run failed", "install_s": install_s, "output": "timed out"}
    run_s = time.monotonic() - started
    lines = [line for line in done.stdout.splitlines() if line.startswith("S1_MJLAB_RESULT ")]
    if done.returncode != 0 or len(lines) != 1:
        return {"status": "run failed", "install_s": install_s, "output": tail(done.stdout + done.stderr)}
    result = json.loads(lines[0].split(" ", 1)[1])
    rows = []
    for measured in result["rows"]:
        row = {"stack": "mjlab", "impl": "mujoco-warp", "batch": measured.pop("num_envs"), "bar": True}
        row["steps_per_s"] = measured.pop("env_steps_per_s", None)
        if "physics_dt" in measured:
            row.update(physics_hz=round(1 / measured["physics_dt"]), control_hz=round(1 / measured["control_dt"]))
        rows.append({**row, **measured})
    return {"status": "measured", "install_s": install_s, "run_s": run_s, **result, "rows": rows}


if RUN_MJLAB:
    try:
        MJLAB = mjlab_section()
    except Exception:  # an optional section records its failure; SMOKE stops on it
        if SMOKE:
            raise
        traceback.print_exc()
        MJLAB = {"status": "failed", "output": traceback.format_exc()}
else:
    MJLAB = {"status": "skipped (RUN_MJLAB = False)"}
MJLAB.setdefault("rows", [])
MJLAB.update(pin=MJLAB_PIN, task=MJLAB_TASK)
RESULTS["mjlab"] = MJLAB
save_results()
print(f"mjlab: {MJLAB['status']}")
if MJLAB["status"] != "measured" and "output" in MJLAB:
    print(f"mjlab {MJLAB['status'].upper()} (optional: the bars rest on Playground)\n{MJLAB['output']}")
for row in MJLAB["rows"]:
    if row.get("error"):
        print(f"  num_envs={row['batch']}: ERROR {row['error'][:400]}")
    else:
        print(
            f"  num_envs={row['batch']}: {row['steps_per_s']:,.0f} env steps/s "
            f"({row['physics_hz']} Hz physics, {row['substeps']} substeps; build {row['build_s']:.0f} s, "
            f"warm-up {row['warmup_s']:.1f} s, timed {row['timed_s']:.1f} s)"
        )

## 4. Single G1 on MuJoCo Playground

`G1JoystickFlatTerrain` inside Brax's training wrappers (episodes,
auto-reset, domain randomization), stepped `UNROLL` times per compiled
call with uniform random actions, for each MJX implementation and batch
size. The overflow check is best effort here: an auto-reset clears a
world's overflow bits.

In [ ]:
from mujoco_playground import registry, wrapper


def measure_playground(impl, batch):
    require_warp_gpu(impl)
    config = registry.get_default_config(PLAYGROUND_ENV)
    overrides = {"impl": impl}
    if impl == "warp":
        overrides["naconmax"] = PLAYGROUND_CONTACTS_PER_ENV * batch
    env = registry.load(PLAYGROUND_ENV, config=config, config_overrides=overrides)
    # Print-only change; Playground's randomization wrapper swaps this same attribute.
    env._mjx_model = quiet(env.mjx_model)
    key = jax.random.PRNGKey(SEED)
    randomization_fn = None
    if PLAYGROUND_DOMAIN_RANDOMIZATION:
        randomizer = registry.get_domain_randomizer(PLAYGROUND_ENV)
        randomization_fn = functools.partial(randomizer, rng=jax.random.split(key, batch))
    training_env = wrapper.wrap_for_brax_training(
        env, episode_length=config.episode_length, action_repeat=1, randomization_fn=randomization_fn
    )

    def unroll(state, key):
        def step(state, key):
            action = jax.random.uniform(key, (batch, env.action_size), minval=-1.0, maxval=1.0)
            return training_env.step(state, action), None

        return jax.lax.scan(step, state, jax.random.split(key, UNROLL))[0]

    started = time.perf_counter()
    state = strong(jax.jit(training_env.reset)(jax.random.split(key, batch)))
    jax.block_until_ready(state)
    reset_s = time.perf_counter() - started
    state, timing = timed(unroll, state)
    steps = batch * UNROLL * timing["timed_calls"]
    return {
        "stack": "playground", "impl": impl, "batch": batch, "bar": True,
        "steps_per_s": steps / timing["timed_s"],
        "physics_steps_per_s": steps * env.n_substeps / timing["timed_s"],
        "env": PLAYGROUND_ENV,
        "control_hz": round(1 / env.dt), "physics_hz": round(1 / env.sim_dt), "timestep_s": env.sim_dt,
        "substeps": env.n_substeps,
        "integrator": mujoco.mjtIntegrator(env.mj_model.opt.integrator).name,
        "solver_iterations": int(env.mj_model.opt.iterations),
        "ls_iterations": int(env.mj_model.opt.ls_iterations),
        "domain_randomization": PLAYGROUND_DOMAIN_RANDOMIZATION,
        "naconmax": overrides.get("naconmax"),
        "unroll": UNROLL, "timed_steps": steps, "reset_s": reset_s, **timing,
        "episodes_ended": int(np.sum(np.asarray(state.info["AutoResetWrapper_done_count"]))),
        **health(state.data),
    }


rows, wall_s = sweep(
    "playground",
    [{"impl": impl, "batch": batch} for impl in PLAYGROUND_IMPLS for batch in PLAYGROUND_BATCH_SIZES],
    measure_playground,
)
RESULTS["playground"] = {"env": PLAYGROUND_ENV, "rows": rows, "wall_s": wall_s}
save_results()
print(f"Playground section: {wall_s / 60:.1f} min")

## 5. Four G1s and a ball

The scene is composed with MjSpec from packaged assets only: four copies
of `assets/robots/unitree_g1_tennis.xml` attached with the prefixes
`a0_`, `a1_`, `b0_` and `b1_` (two per side, the right-wrist racket
included), and a free ball whose geom copies `ball_geom` from
`assets/humanoid_tennis.xml`. The floor is a plane with that court
surface's contact parameters, and the ball-floor pair copies its
`ball_court_pair`. Robot-robot contacts stay masked by the model's own
contact categories, as in the two-player env.

The simplified collision set, used only when MJX rejects the full one
(or with `SIMPLIFY_COLLISION = "always"`), deletes the visual meshes
(contype 0: they never collide), turns ellipsoids (the racket
stringbeds) into boxes of the same half-extents, and drops the
ball-floor pair from condim 6 to 3 (no rolling friction).

In [ ]:
import mujoco
import numpy as np

from courtside_dynamics.assets import asset_path
from courtside_dynamics.envs.robot_models import UNITREE_G1_STAND_BASE_HEIGHT, get_robot_model_spec

G1 = get_robot_model_spec("unitree_g1")
# Doubles: team A faces +x, team B faces -x, partners 4 m apart.
SEATS = {
    "a0": ((-6.0, -2.0, 0.0), (1.0, 0.0, 0.0, 0.0)),
    "a1": ((-6.0, 2.0, 0.0), (1.0, 0.0, 0.0, 0.0)),
    "b0": ((6.0, -2.0, 0.0), (0.0, 0.0, 0.0, 1.0)),
    "b1": ((6.0, 2.0, 0.0), (0.0, 0.0, 0.0, 1.0)),
}
BALL_POS = (-5.5, 0.0, 1.3)
BALL_VEL = (8.0, 0.0, 3.0)  # a lob from team A toward team B
INTEGRATORS = {
    "euler": mujoco.mjtIntegrator.mjINT_EULER,
    "implicitfast": mujoco.mjtIntegrator.mjINT_IMPLICITFAST,
    "rk4": mujoco.mjtIntegrator.mjINT_RK4,
}
# Contact fields copied from the packaged court's geoms, so the scene's
# ball and floor cannot drift from the env's.
CONTACT_FIELDS = (
    "contype", "conaffinity", "condim", "priority", "friction", "solmix", "solref", "solimp", "margin", "gap",
)
BALL_FIELDS = (*CONTACT_FIELDS, "type", "size", "mass", "typeinertia", "rgba")


def copy_fields(target, source, fields):
    for field in fields:
        setattr(target, field, getattr(source, field))
    return target


def simplify(spec):
    """Apply the simplified collision set to one robot spec; returns what changed."""
    meshes = [geom for geom in spec.geoms if geom.type == mujoco.mjtGeom.mjGEOM_MESH]
    for geom in meshes:
        if geom.contype or geom.conaffinity:
            raise ValueError(f"mesh geom {geom.name!r} collides: the simplified set only drops visual meshes")
        spec.delete(geom)
    for mesh in list(spec.meshes):
        spec.delete(mesh)
    ellipsoids = [geom for geom in spec.geoms if geom.type == mujoco.mjtGeom.mjGEOM_ELLIPSOID]
    for geom in ellipsoids:
        geom.type = mujoco.mjtGeom.mjGEOM_BOX
    return [
        f"deleted {len(meshes)} visual mesh geoms and their meshes",
        "ellipsoid -> box, same half-extents: " + ", ".join(geom.name for geom in ellipsoids),
    ]


def build_scene(physics_hz, *, simplified=False):
    """Compile four packaged G1s and the packaged ball over a plane court.

    Returns the model and the simplifications applied (empty for the full set)."""
    court = mujoco.MjSpec.from_file(asset_path("humanoid_tennis.xml"))
    spec = mujoco.MjSpec()
    spec.modelname = "s1_four_g1_ball"
    spec.option.timestep = 1.0 / physics_hz
    spec.option.integrator = INTEGRATORS[INTEGRATOR]
    spec.option.iterations = SOLVER_ITERATIONS
    spec.option.ls_iterations = LS_ITERATIONS
    copy_fields(
        spec.worldbody.add_geom(name="floor", type=mujoco.mjtGeom.mjGEOM_PLANE, size=[0, 0, 0.05]),
        court.geom("court_surface"),
        CONTACT_FIELDS,
    )
    changes = []
    for seat, (pos, quat) in SEATS.items():
        robot = mujoco.MjSpec.from_file(asset_path(G1.mjcf_asset))
        # The composite runs on the parent's options; matching them here
        # only silences MjSpec's attach-conflict warning.
        robot.option.timestep = spec.option.timestep
        robot.option.integrator = spec.option.integrator
        robot.option.iterations = spec.option.iterations
        robot.option.ls_iterations = spec.option.ls_iterations
        if simplified:
            changes += [f"{seat}: {change}" for change in simplify(robot)]
        frame = spec.worldbody.add_frame(name=f"{seat}_spawn", pos=pos, quat=quat)
        frame.attach_body(robot.body("pelvis"), f"{seat}_", "")
    ball = spec.worldbody.add_body(name="ball", pos=BALL_POS)
    ball.add_freejoint(name="ball_free")
    copy_fields(ball.add_geom(name="ball_geom"), court.geom("ball_geom"), BALL_FIELDS)
    [court_pair] = [pair for pair in court.pairs if pair.name == "ball_court_pair"]
    condim = court_pair.condim
    if simplified and condim > 3:
        changes.append(f"ball-floor pair condim {condim} -> 3")
        condim = 3
    spec.add_pair(
        name="ball_floor_pair", geomname1="ball_geom", geomname2="floor", condim=condim,
        friction=court_pair.friction, solref=court_pair.solref, solreffriction=court_pair.solreffriction,
        solimp=court_pair.solimp, margin=court_pair.margin, gap=court_pair.gap,
    )
    return spec.compile(), changes


def scene_home(model):
    """qpos, qvel and ctrl: every seat at the registry's standing reference, the ball lobbed."""
    qpos, qvel, ctrl = model.qpos0.copy(), np.zeros(model.nv), np.zeros(model.nu)
    for seat in SEATS:
        base = int(model.joint(f"{seat}_floating_base_joint").qposadr[0])
        qpos[base + 2] = UNITREE_G1_STAND_BASE_HEIGHT  # x, y and heading come from the spawn frame
        for name, target in zip(G1.actuator_names, G1.stand_joint_positions, strict=True):
            qpos[int(model.joint(f"{seat}_{name}").qposadr[0])] = target
            ctrl[model.actuator(f"{seat}_{name}").id] = target
    ball = int(model.joint("ball_free").dofadr[0])
    qvel[ball : ball + 3] = BALL_VEL
    return qpos, qvel, ctrl


def describe(model):
    types = np.unique(model.geom_type, return_counts=True)
    return {
        "nq": model.nq, "nv": model.nv, "nu": model.nu, "nbody": model.nbody, "ngeom": model.ngeom,
        "geom_types": {mujoco.mjtGeom(kind).name.removeprefix("mjGEOM_").lower(): int(n) for kind, n in zip(*types)},
        "colliding_geoms": int(np.sum((model.geom_contype != 0) | (model.geom_conaffinity != 0))),
        "nmesh": model.nmesh, "npair": model.npair, "nsensor": model.nsensor,
        "timestep_s": model.opt.timestep, "integrator": INTEGRATOR,
        "solver_iterations": model.opt.iterations, "ls_iterations": model.opt.ls_iterations,
    }

In [ ]:
def scene_model(impl, physics_hz):
    """The scene on MJX: the full collision set, unless MJX rejects it."""

    def convert(model):
        mx = mjx.put_model(model, impl=impl)
        # Some features fail only when traced: trace one step, without compiling.
        capacity = {"naconmax": SCENE_CONTACTS_PER_WORLD, "njmax": SCENE_CONSTRAINTS_PER_WORLD} if impl == "warp" else {}
        jax.eval_shape(functools.partial(mjx.step, mx), mjx.make_data(model, impl=impl, **capacity))
        return mx

    rejection = None
    if SIMPLIFY_COLLISION == "auto":
        model, _ = build_scene(physics_hz)
        try:
            return model, convert(model), "full", None, []
        except NotImplementedError as error:
            rejection = f"{type(error).__name__}: {error}"
            print(f"MJX ({impl}) REJECTED the full collision set: {rejection}\nFalling back to the simplified set.")
    model, changes = build_scene(physics_hz, simplified=True)
    return model, convert(model), "simplified", rejection, changes


def measure_scene(impl, batch, physics_hz):
    require_warp_gpu(impl)
    model, mx, collision_set, rejection, changes = scene_model(impl, physics_hz)
    mx = quiet(mx)
    substeps = physics_hz // CONTROL_HZ
    qpos, qvel, ctrl = (jnp.asarray(array) for array in scene_home(model))
    low, high = (jnp.asarray(bound) for bound in model.actuator_ctrlrange.T)
    capacity = {}
    if impl == "warp":
        capacity = {"naconmax": SCENE_CONTACTS_PER_WORLD * batch, "njmax": SCENE_CONSTRAINTS_PER_WORLD}

    def init(_):
        return mjx.make_data(model, impl=impl, **capacity).replace(qpos=qpos, qvel=qvel, ctrl=ctrl)

    def control_step(data, key):
        noise = jax.random.uniform(key, ctrl.shape, minval=-1.0, maxval=1.0)
        data = data.replace(ctrl=jnp.clip(ctrl + ACTION_SCALE * noise, low, high))
        return jax.lax.scan(lambda data, _: (mjx.step(mx, data), None), data, None, length=substeps)[0]

    def unroll(data, key):
        def body(data, key):
            return jax.vmap(control_step)(data, jax.random.split(key, batch)), None

        return jax.lax.scan(body, data, jax.random.split(key, SCENE_UNROLL))[0]

    data = strong(jax.jit(jax.vmap(init))(jnp.arange(batch)))
    data, timing = timed(unroll, data)
    control_steps = batch * SCENE_UNROLL * timing["timed_calls"]
    rate = control_steps / timing["timed_s"]
    heights = np.asarray(data.qpos)[:, [int(model.joint(f"{seat}_floating_base_joint").qposadr[0]) + 2 for seat in SEATS]]
    return {
        "stack": "mjx-scene", "impl": impl, "batch": batch, "bar": physics_hz == PHYSICS_HZ,
        "steps_per_s": rate, "physics_steps_per_s": rate * substeps, "agent_steps_per_s": rate * len(SEATS),
        "physics_hz": physics_hz, "timestep_s": 1.0 / physics_hz, "control_hz": CONTROL_HZ, "substeps": substeps,
        "integrator": INTEGRATOR, "solver_iterations": SOLVER_ITERATIONS, "ls_iterations": LS_ITERATIONS,
        "collision_set": collision_set, "mjx_rejection": rejection, "simplifications": changes,
        "naconmax": capacity.get("naconmax"), "njmax": capacity.get("njmax"),
        "unroll": SCENE_UNROLL, "timed_control_steps": control_steps, **timing,
        "fallen_fraction": float(np.mean(heights < 0.5)),
        **health(data),
    }


COMPOSITION, _ = build_scene(PHYSICS_HZ)
RESULTS["scene"] = {"composition": describe(COMPOSITION), "rows": []}
print(json.dumps(RESULTS["scene"]["composition"], indent=2))
rows, wall_s = sweep(
    "scene",
    [
        {"impl": impl, "batch": batch, "physics_hz": PHYSICS_HZ}
        for impl in SCENE_IMPLS
        for batch in SCENE_BATCH_SIZES
    ],
    measure_scene,
)
best = best_row(rows)
if best is not None and EXTRA_PHYSICS_HZ:
    extra, extra_s = sweep(
        "scene",
        [{"impl": best["impl"], "batch": best["batch"], "physics_hz": hz} for hz in EXTRA_PHYSICS_HZ],
        measure_scene,
    )
    rows, wall_s = rows + extra, wall_s + extra_s
RESULTS["scene"].update(rows=rows, wall_s=wall_s)
save_results()
print(f"Scene section: {wall_s / 60:.1f} min")

## 6. Results

Every row, one line each; the JSON (saved after every section) holds
the full rows, the settings and the environment record.

In [ ]:
def summary(row):
    if row.get("error"):
        return f"ERROR {row['error'][:120]}"
    notes = []
    if not row.get("bar"):
        notes.append("informational")
    if row.get("overflow"):
        notes.append(f"OVERFLOW {row['overflow']}")
    if row.get("nonfinite_worlds"):
        notes.append(f"NON-FINITE {row['nonfinite_worlds']} worlds")
    if row.get("collision_set") == "simplified":
        notes.append("simplified collision set")
    return f"{row['steps_per_s']:>12,.0f} steps/s  " + ("; ".join(notes) if notes else "counts")


ALL_ROWS = MJLAB["rows"] + RESULTS["playground"]["rows"] + RESULTS["scene"]["rows"]
print(f"{'stack':<11} {'impl':<12} {'batch':>6} {'physics':>8} {'substeps':>8}")
for row in ALL_ROWS:
    physics = f"{row['physics_hz']} Hz" if row.get("physics_hz") else "-"
    print(
        f"{row.get('stack', '-'):<11} {row.get('impl', '-'):<12} {row.get('batch', '-'):>6} "
        f"{physics:>8} {row.get('substeps', '-')!s:>8}  {summary(row)}"
    )
print("Saved", save_results())

## 7. Verdict

PASS or FAIL per bar, from each bar's best counting row, and the
design's next step. A SMOKE run prints no verdict. A bar with no valid
measurement raises after the results are saved.

In [ ]:
VERDICT = s1_verdict(
    MJLAB["rows"] + RESULTS["playground"]["rows"],
    RESULTS["scene"]["rows"],
    single_bar=SINGLE_G1_BAR,
    four_bar=FOUR_G1_BAR,
    gpu_name=GPU_NAME,
    smoke=SMOKE,
)
RESULTS["verdict"] = VERDICT
RESULTS["finished_utc"] = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
path = save_results()
print(f"S1 on {GPU_NAME}" + (" (SMOKE: code paths only)" if SMOKE else ""))
for key, label, unit in (
    ("single_g1", "Single G1", "env steps/s"),
    ("four_g1", "4xG1 + ball", f"control steps/s at {PHYSICS_HZ} Hz physics"),
):
    entry = VERDICT["bars"][key]
    best = entry["best"]
    if best is None:
        measured = "no valid measurement"
    else:
        measured = (
            f"{best['steps_per_s']:,.0f} {unit} ({best['stack']}, {best['impl']}, batch {best['batch']}"
            + (", simplified collision set" if best.get("collision_set") == "simplified" else "")
            + ")"
        )
    print(f"{entry['status']:<9} {label}: {measured}; bar {entry['bar']:,.0f}")
if VERDICT["excluded_rows"]:
    print(f"{VERDICT['excluded_rows']} bar configuration(s) did not count: error, skip, overflow or non-finite state (section 6)")
print(f"Overall: {VERDICT['overall']}")
print(f"Next step: {VERDICT['next_step']}")
print(f"Results: {path} ({RESULTS['wall_s'] / 60:.0f} min)")
if VERDICT["overall"] == "NO RESULT":
    raise RuntimeError("S1 has no verdict: a bar has no valid measurement (see the rows above)")